
# ViT Family — Reference Notebook

This notebook is a **single reference implementation + explanation** for every model in the
"ViT — Priority List" you compiled. It is meant to be the thing you re-open when you forget
*why* DINOv2 matters, or *what* actually changes between CLIP and SigLIP.

**How this notebook is organized** — one section per model, each with:
1. **What it is / why it matters / where it's used today** (markdown)
2. **A from-scratch, minimal, readable PyTorch implementation** of the idea that made that
   model important (not a production reimplementation — a *teaching* implementation)
3. A tiny demo forward pass so you can see the shapes flow through

## Roadmap covered here

```
01. ViT              -> Transformer vision fundamentals (patchify + self-attention)
02. DINOv2            -> Self-supervised representation learning (student/teacher distillation)
03. CLIP              -> Vision-language contrastive alignment (softmax InfoNCE)
04. SigLIP / SigLIP2   -> Vision-language alignment, sigmoid loss (scales better, simpler)
05. Swin Transformer  -> Hierarchical vision, windowed + shifted attention
06. DETR / Def-DETR   -> Transformer object detection, set prediction + bipartite matching
07. MAE               -> Masked autoencoding pretraining
08. BEiT              -> Masked *token* modeling (BERT-style, but for images)
09. SAM / SAM 2       -> Promptable segmentation (image encoder + prompt encoder + mask decoder)
10. ViViT / Video ViT -> Spatiotemporal transformers (tubelets, factorized attention)
```

**"If you only learn 5":** ViT → DINOv2 → SigLIP → Swin → SAM 2. Everything else builds on
ideas introduced in those five.

> Note: this notebook is written to be **run in an environment with `torch` installed**
> (`pip install torch`). Shapes and logic have been hand-verified; run cells top-to-bottom.


In [3]:

# If needed:
# !pip install torch --quiet

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda



## 1. ViT (Vision Transformer) — ⭐⭐⭐⭐⭐ Foundation

**Idea:** split an image into fixed-size patches, linearly project each patch into a token
embedding (exactly like word embeddings), prepend a learnable `[CLS]` token, add learned/
sinusoidal position embeddings, and run the whole sequence through a standard Transformer
encoder. Classification uses the `[CLS]` token's output embedding.

**Why it matters:** it proved that a plain Transformer — with *no* convolutional inductive
bias — can match or beat CNNs on image classification given enough data. Every model below
is either a variant of ViT or uses a ViT as a backbone.

**Used today for:** the backbone of nearly every modern vision/multimodal model (DINOv2,
CLIP, SigLIP, SAM, and the "ViT" patchify step is also how DiT treats image latents).


In [4]:

class PatchEmbed(nn.Module):
    """Splits an image into non-overlapping patches and linearly embeds each one."""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=384):
        super().__init__()
        assert img_size % patch_size == 0
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2
        # A strided conv with kernel == stride == patch_size IS "cut into patches + linear project"
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)                      # (B, embed_dim, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)       # (B, num_patches, embed_dim)
        return x


class MHSA(nn.Module):
    """Standard multi-head self-attention."""
    def __init__(self, dim, num_heads=6, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                          # each (B, heads, N, head_dim)
        attn = (q @ k.transpose(-2, -1)) * self.scale             # (B, heads, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(out))


class MLP(nn.Module):
    def __init__(self, dim, hidden_ratio=4.0, drop=0.0):
        super().__init__()
        hidden = int(dim * hidden_ratio)
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, dim), nn.Dropout(drop),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-norm Transformer encoder block: x + Attn(LN(x)), then x + MLP(LN(x))."""
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MHSA(dim, num_heads, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, mlp_ratio, drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000,
                 embed_dim=384, depth=12, num_heads=6, mlp_ratio=4.0):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes) if num_classes > 0 else nn.Identity()

    def forward_features(self, x):
        x = self.patch_embed(x)                                   # (B, N, D)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x                                                   # (B, N+1, D)  -- patch tokens + CLS

    def forward(self, x):
        feats = self.forward_features(x)
        cls_out = feats[:, 0]                                      # CLS token representation
        return self.head(cls_out)


# --- demo ---
vit = ViT(img_size=224, patch_size=16, embed_dim=192, depth=4, num_heads=3, num_classes=10)
dummy_imgs = torch.randn(2, 3, 224, 224)
logits = vit(dummy_imgs)
print("ViT logits:", logits.shape)   # (2, 10)


ViT logits: torch.Size([2, 10])



## 2. DINOv2 — ⭐⭐⭐⭐⭐ Self-supervised visual representations

**Idea:** train a ViT with **no labels** using self-distillation. A *student* network and a
slowly-updated (EMA / momentum) *teacher* network see different augmented crops of the same
image. The student is trained to match the teacher's output distribution over a set of
"prototypes" (a softmax over a learned projection head), even though it only sees a local
crop while the teacher sees a global one. Over training, the student learns semantically rich
features without ever seeing a label. DINOv2 additionally adds: a fixed high-resolution
teacher training phase, better data curation, and (in follow-ups) "register tokens" to fix
attention-map artifacts.

**Why it matters:** the frozen patch/CLS features it produces transfer extremely well to
*many* downstream tasks — often via a simple linear probe or k-NN, with **no fine-tuning**.

**Used today for:** retrieval, dense/segmentation features, depth estimation, robotics
perception, geospatial imagery, and as an off-the-shelf visual feature extractor everywhere
you'd have used a supervised CNN backbone a few years ago.


In [5]:

class DINOHead(nn.Module):
    """Projects the backbone's CLS embedding into a (large) prototype space."""
    def __init__(self, in_dim, out_dim=2048, hidden_dim=1024):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.mlp(x)


class DINO(nn.Module):
    """
    Minimal student/teacher self-distillation wrapper around the ViT above.
    Real DINOv2 uses multi-crop (2 global + several local crops); we show the
    core mechanism with just one student view and one teacher view.
    """
    def __init__(self, embed_dim=192, out_dim=1024, momentum=0.996,
                 student_temp=0.1, teacher_temp=0.04):
        super().__init__()
        self.student_backbone = ViT(embed_dim=embed_dim, depth=4, num_heads=3, num_classes=0)
        self.teacher_backbone = ViT(embed_dim=embed_dim, depth=4, num_heads=3, num_classes=0)
        self.student_head = DINOHead(embed_dim, out_dim)
        self.teacher_head = DINOHead(embed_dim, out_dim)

        # teacher starts identical to student and is NEVER trained by gradient descent
        self.teacher_backbone.load_state_dict(self.student_backbone.state_dict())
        self.teacher_head.load_state_dict(self.student_head.state_dict())
        for p in self.teacher_backbone.parameters():
            p.requires_grad = False
        for p in self.teacher_head.parameters():
            p.requires_grad = False

        self.m = momentum
        self.student_temp = student_temp
        self.teacher_temp = teacher_temp
        self.register_buffer("center", torch.zeros(1, out_dim))

    @torch.no_grad()
    def update_teacher(self):
        """EMA update: teacher = m * teacher + (1-m) * student."""
        for t_p, s_p in zip(self.teacher_backbone.parameters(), self.student_backbone.parameters()):
            t_p.data.mul_(self.m).add_(s_p.data, alpha=1 - self.m)
        for t_p, s_p in zip(self.teacher_head.parameters(), self.student_head.parameters()):
            t_p.data.mul_(self.m).add_(s_p.data, alpha=1 - self.m)

    @torch.no_grad()
    def update_center(self, teacher_out, momentum=0.9):
        """Centering prevents the teacher from collapsing to a single prototype."""
        batch_center = teacher_out.mean(dim=0, keepdim=True)
        self.center.mul_(momentum).add_(batch_center, alpha=1 - momentum)

    def dino_loss(self, student_view, teacher_view):
        s_cls = self.student_backbone.forward_features(student_view)[:, 0]
        s_out = self.student_head(s_cls) / self.student_temp

        with torch.no_grad():
            t_cls = self.teacher_backbone.forward_features(teacher_view)[:, 0]
            t_out = self.teacher_head(t_cls)
            t_out = F.softmax((t_out - self.center) / self.teacher_temp, dim=-1)  # sharpen + center

        loss = -(t_out * F.log_softmax(s_out, dim=-1)).sum(dim=-1).mean()  # cross-entropy
        self.update_center(t_out)
        return loss


# --- demo: one "training step" ---
dino = DINO(embed_dim=192, out_dim=256)
global_crop = torch.randn(4, 3, 224, 224)   # teacher sees the "easier" global view
local_crop = torch.randn(4, 3, 224, 224)    # student sees a harder/local view
loss = dino.dino_loss(student_view=local_crop, teacher_view=global_crop)
loss.backward()
dino.update_teacher()   # teacher never gets gradients, only EMA updates
print("DINO self-distillation loss:", loss.item())


DINO self-distillation loss: 4.9607038497924805



## 3. CLIP — ⭐⭐⭐⭐ Vision-language contrastive alignment

**Idea:** train an image encoder and a text encoder jointly so that matching (image, caption)
pairs have **high cosine similarity** and non-matching pairs have low similarity. This is done
with a **symmetric InfoNCE / softmax contrastive loss** over an entire batch: for each image,
its true caption should score highest among *all* captions in the batch (and vice versa).

**Why it matters:** it decouples vision from fixed label sets entirely — you can do zero-shot
classification by just embedding class-name text prompts and comparing to the image
embedding. It's the ancestor of essentially every modern VLM's vision-text alignment stage.

**Used today for:** zero-shot classification, image/text retrieval, and as the "vision
encoder" glue in many multimodal LLMs (superseded in a lot of pipelines by SigLIP — see next).


In [6]:

class SimpleTextEncoder(nn.Module):
    """A tiny causal-ish Transformer text encoder (stand-in for CLIP's text tower)."""
    def __init__(self, vocab_size=10000, max_len=32, embed_dim=192, depth=4, num_heads=3):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_len, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, token_ids):
        x = self.tok_embed(token_ids) + self.pos_embed[:, :token_ids.shape[1]]
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x.mean(dim=1)   # pool over sequence -> (B, D)  (CLIP actually uses EOS-token pooling)


class CLIP(nn.Module):
    def __init__(self, embed_dim=192, proj_dim=128):
        super().__init__()
        self.image_encoder = ViT(embed_dim=embed_dim, depth=4, num_heads=3, num_classes=0)
        self.text_encoder = SimpleTextEncoder(embed_dim=embed_dim, depth=4, num_heads=3)
        self.image_proj = nn.Linear(embed_dim, proj_dim, bias=False)
        self.text_proj = nn.Linear(embed_dim, proj_dim, bias=False)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))  # learned temperature

    def encode_image(self, images):
        feats = self.image_encoder.forward_features(images)[:, 0]
        return F.normalize(self.image_proj(feats), dim=-1)

    def encode_text(self, token_ids):
        feats = self.text_encoder(token_ids)
        return F.normalize(self.text_proj(feats), dim=-1)

    def forward(self, images, token_ids):
        img_emb = self.encode_image(images)
        txt_emb = self.encode_text(token_ids)
        logits = self.logit_scale.exp() * img_emb @ txt_emb.t()   # (B, B) similarity matrix
        return logits

    def clip_loss(self, images, token_ids):
        logits = self.forward(images, token_ids)                 # (B, B)
        targets = torch.arange(logits.shape[0], device=logits.device)  # diagonal = correct pairs
        loss_i2t = F.cross_entropy(logits, targets)               # softmax over columns
        loss_t2i = F.cross_entropy(logits.t(), targets)           # softmax over rows
        return (loss_i2t + loss_t2i) / 2


# --- demo ---
clip_model = CLIP(embed_dim=192, proj_dim=128)
imgs = torch.randn(4, 3, 224, 224)
caption_tokens = torch.randint(0, 10000, (4, 16))
loss = clip_model.clip_loss(imgs, caption_tokens)
print("CLIP contrastive (softmax) loss:", loss.item())


CLIP contrastive (softmax) loss: 1.8532209396362305



## 4. SigLIP / SigLIP 2 — ⭐⭐⭐⭐⭐ Image ↔ text understanding

**Idea:** replace CLIP's batch-wide softmax contrastive loss with a **per-pair sigmoid loss**.
Every (image, text) pair in the batch is treated as an *independent binary classification*
problem ("does this pair match? yes/no") instead of needing a normalized distribution over
the whole batch. This removes the need to compute a global softmax normalizer, which means
you don't need a giant batch size (or cross-device synchronization of it) to get a good
learning signal — it's simpler and more compute/memory-efficient at scale.

SigLIP 2 layers on additional pretraining objectives (self-distillation similar to DINO, and
masked-prediction similar to MAE/BEiT) on top of the sigmoid contrastive backbone.

**Why it matters:** better accuracy/compute trade-off than CLIP, and it has become a default
choice as the vision tower in modern VLMs.

**Used today for:** VLMs (vision encoder), image search, multimodal retrieval.


In [7]:

class SigLIP(CLIP):
    """Reuses the CLIP towers -- SigLIP's innovation is purely in the *loss*, not the encoders."""
    def __init__(self, embed_dim=192, proj_dim=128):
        super().__init__(embed_dim, proj_dim)
        self.logit_bias = nn.Parameter(torch.tensor(-10.0))  # SigLIP adds a learned bias term

    def siglip_loss(self, images, token_ids):
        img_emb = self.encode_image(images)
        txt_emb = self.encode_text(token_ids)
        B = img_emb.shape[0]

        logits = self.logit_scale.exp() * img_emb @ txt_emb.t() + self.logit_bias  # (B, B)
        # labels: +1 on the diagonal (matching pair), -1 everywhere else (non-matching pair)
        labels = 2 * torch.eye(B, device=logits.device) - 1
        # every entry is an independent binary (sigmoid) classification -> no batch-wide softmax
        loss = -F.logsigmoid(labels * logits).mean()
        return loss


# --- demo: same encoders, different (cheaper, more scalable) loss ---
siglip_model = SigLIP(embed_dim=192, proj_dim=128)
loss = siglip_model.siglip_loss(imgs, caption_tokens)
print("SigLIP sigmoid loss:", loss.item())
print("Contrast with CLIP: SigLIP's loss is O(B) independent decisions, not one O(B)-way softmax.")


SigLIP sigmoid loss: 2.536432981491089
Contrast with CLIP: SigLIP's loss is O(B) independent decisions, not one O(B)-way softmax.



## 5. Swin Transformer — ⭐⭐⭐⭐⭐ Hierarchical vision + efficient attention

**Idea:** plain ViT does full global self-attention over all patches — quadratic cost in
number of patches, and it produces a single-resolution feature map, which is awkward for
dense prediction (detection/segmentation) where you want a feature pyramid like a CNN's.
Swin fixes both:
- **Windowed attention**: split the feature map into small non-overlapping windows and do
  self-attention *only within each window* (linear cost in image size).
- **Shifted windows**: alternate layers shift the window grid by half a window, so information
  can flow *across* window boundaries over successive layers.
- **Patch merging**: periodically merge 2×2 neighboring patches to downsample, like CNN
  pooling — building a hierarchical, multi-resolution feature pyramid.

**Why it matters:** gives ViT-style modeling CNN-like properties (locality + hierarchy)
needed for dense vision tasks, at much lower compute than global attention.

**Used today for:** detection, segmentation, and any high-resolution vision task where
full quadratic attention is too expensive.


In [8]:

def window_partition(x, window_size):
    """(B, H, W, C) -> (num_windows*B, window_size, window_size, C)"""
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class SwinBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size=7, shift_size=0, mlp_ratio=4.0):
        super().__init__()
        self.window_size = window_size
        self.shift_size = shift_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MHSA(dim, num_heads)   # windowed attention reuses plain MHSA *within* a window
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, mlp_ratio)

    def forward(self, x, H, W):
        B, N, C = x.shape
        shortcut = x
        x = self.norm1(x).view(B, H, W, C)

        if self.shift_size > 0:
            # cyclic shift lets information cross window borders on alternating layers
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))

        windows = window_partition(x, self.window_size)                 # (nW*B, ws, ws, C)
        windows = windows.view(-1, self.window_size * self.window_size, C)
        attn_windows = self.attn(windows)                               # self-attn INSIDE each window only
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        x = window_reverse(attn_windows, self.window_size, H, W)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))

        x = x.view(B, H * W, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x


class PatchMerging(nn.Module):
    """Downsamples 2x2 neighboring patches -> doubles channel dim, like CNN pooling + channel expand."""
    def __init__(self, dim):
        super().__init__()
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = nn.LayerNorm(4 * dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.view(B, H, W, C)
        x0, x1, x2, x3 = x[:, 0::2, 0::2], x[:, 1::2, 0::2], x[:, 0::2, 1::2], x[:, 1::2, 1::2]
        x = torch.cat([x0, x1, x2, x3], dim=-1).view(B, -1, 4 * C)
        return self.reduction(self.norm(x)), H // 2, W // 2


# --- demo: one Swin stage (regular window block -> shifted window block -> downsample) ---
C, H, W = 96, 56, 56
x = torch.randn(2, H * W, C)
block1 = SwinBlock(C, num_heads=3, window_size=7, shift_size=0)
block2 = SwinBlock(C, num_heads=3, window_size=7, shift_size=3)   # shift = window_size // 2
merge = PatchMerging(C)

x = block1(x, H, W)
x = block2(x, H, W)
x, H, W = merge(x, H, W)
print("after one Swin stage:", x.shape, "| new grid:", H, "x", W)


after one Swin stage: torch.Size([2, 784, 192]) | new grid: 28 x 28



## 6. DETR / Deformable DETR — ⭐⭐⭐⭐ Transformer object detection

**Idea:** treat object detection as **direct set prediction**. A CNN/ViT backbone produces
image features; a Transformer encoder contextualizes them; a Transformer **decoder** takes a
fixed number of learned "object query" embeddings and cross-attends into the encoded image
features, and each query directly outputs one (class, bounding box) prediction. Training uses
**bipartite (Hungarian) matching** between the fixed set of predictions and the ground-truth
objects, so each prediction is optimized against its best-matched target — no anchors, no
NMS, no hand-designed proposal generation.

*Deformable DETR* fixes DETR's slow convergence and poor small-object performance by
replacing full attention over all pixels with **deformable attention** — each query only
attends to a small, learned set of sampling points near a reference location, which is much
cheaper and converges much faster.

**Why it matters:** first fully end-to-end, NMS-free Transformer detector.

**Used today for:** object detection pipelines built on Transformers (and the "decoder +
learned queries + Hungarian matching" pattern reappears in DETR-style segmentation too).


In [9]:

class DETRDecoderLayer(nn.Module):
    """Self-attention over queries + cross-attention from queries into encoder memory."""
    def __init__(self, dim, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.norm3 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, mlp_ratio)

    def forward(self, queries, memory):
        q = self.norm1(queries)
        queries = queries + self.self_attn(q, q, q)[0]
        q = self.norm2(queries)
        queries = queries + self.cross_attn(q, memory, memory)[0]   # attend INTO the image features
        queries = queries + self.mlp(self.norm3(queries))
        return queries


class DETR(nn.Module):
    def __init__(self, embed_dim=192, num_queries=20, num_classes=10, dec_depth=3):
        super().__init__()
        self.backbone = ViT(embed_dim=embed_dim, depth=4, num_heads=3, num_classes=0)  # encoder side
        self.query_embed = nn.Parameter(torch.zeros(1, num_queries, embed_dim))
        nn.init.trunc_normal_(self.query_embed, std=0.02)
        self.decoder_layers = nn.ModuleList(
            [DETRDecoderLayer(embed_dim, num_heads=3) for _ in range(dec_depth)]
        )
        self.class_head = nn.Linear(embed_dim, num_classes + 1)  # +1 for "no object"
        self.box_head = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.ReLU(), nn.Linear(embed_dim, 4))

    def forward(self, images):
        memory = self.backbone.forward_features(images)[:, 1:]   # drop CLS, keep patch tokens as "memory"
        queries = self.query_embed.expand(images.shape[0], -1, -1)
        for layer in self.decoder_layers:
            queries = layer(queries, memory)
        class_logits = self.class_head(queries)          # (B, num_queries, num_classes+1)
        boxes = self.box_head(queries).sigmoid()          # (B, num_queries, 4)  normalized cxcywh
        return class_logits, boxes


def hungarian_match(pred_logits, pred_boxes, tgt_classes, tgt_boxes):
    """
    One image's bipartite matching between N predictions and M ground-truth objects,
    using a simple cost = classification cost + L1 box cost (DETR's real cost also
    includes a GIoU term). Falls back to greedy matching if scipy isn't available.
    """
    with torch.no_grad():
        probs = pred_logits.softmax(-1)                                   # (N, C+1)
        class_cost = -probs[:, tgt_classes]                               # (N, M)
        box_cost = torch.cdist(pred_boxes, tgt_boxes, p=1)                # (N, M)
        cost = class_cost + box_cost
        try:
            from scipy.optimize import linear_sum_assignment
            row_ind, col_ind = linear_sum_assignment(cost.cpu().numpy())
        except ImportError:
            # greedy fallback: repeatedly pick the globally-cheapest remaining pair
            row_ind, col_ind = [], []
            remaining_rows = list(range(cost.shape[0]))
            remaining_cols = list(range(cost.shape[1]))
            c = cost.clone()
            for _ in range(min(cost.shape)):
                idx = torch.argmin(c[remaining_rows][:, remaining_cols])
                r = remaining_rows[idx // len(remaining_cols)]
                col = remaining_cols[idx % len(remaining_cols)]
                row_ind.append(r); col_ind.append(col)
                remaining_rows.remove(r); remaining_cols.remove(col)
        return row_ind, col_ind


# --- demo ---
detr = DETR(embed_dim=192, num_queries=10, num_classes=5)
class_logits, boxes = detr(torch.randn(1, 3, 224, 224))
print("DETR class_logits:", class_logits.shape, "| boxes:", boxes.shape)

# fake ground truth: 3 objects in this image
tgt_classes = torch.tensor([1, 3, 0])
tgt_boxes = torch.rand(3, 4)
rows, cols = hungarian_match(class_logits[0], boxes[0], tgt_classes, tgt_boxes)
print("matched prediction indices:", rows, "-> ground-truth indices:", cols)


DETR class_logits: torch.Size([1, 10, 6]) | boxes: torch.Size([1, 10, 4])
matched prediction indices: [4 5 6] -> ground-truth indices: [1 2 0]



## 7. MAE (Masked Autoencoder) — ⭐⭐⭐⭐ Masked visual pretraining

**Idea:** mask out a **very high fraction** (e.g. 75%) of image patches, feed *only the
visible patches* through a (large) ViT encoder, then hand the encoded visible tokens plus
mask placeholder tokens to a lightweight decoder that reconstructs the raw pixel values of
the masked patches. Because the encoder never processes mask tokens, it's very compute
efficient relative to the mask ratio.

**Why it matters:** shows that a very simple pixel-reconstruction objective, with an
aggressive masking ratio, is enough to learn strong transferable representations —
no negative pairs, no augmentation-invariance tricks, no momentum teacher required (unlike
DINO/SimCLR-style methods).

**Used today for:** representation-learning pretraining stage, especially where you want a
compute-cheap self-supervised objective before supervised fine-tuning.


In [10]:

class MAE(nn.Module):
    def __init__(self, img_size=224, patch_size=16, embed_dim=192, decoder_dim=96,
                 enc_depth=4, dec_depth=2, num_heads=3, mask_ratio=0.75):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, embed_dim=embed_dim)
        self.patch_size = patch_size
        self.num_patches = self.patch_embed.num_patches
        self.mask_ratio = mask_ratio

        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.encoder = nn.ModuleList([TransformerBlock(embed_dim, num_heads) for _ in range(enc_depth)])
        self.enc_norm = nn.LayerNorm(embed_dim)

        self.decoder_embed = nn.Linear(embed_dim, decoder_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, decoder_dim))
        nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)
        self.decoder = nn.ModuleList([TransformerBlock(decoder_dim, num_heads) for _ in range(dec_depth)])
        self.dec_norm = nn.LayerNorm(decoder_dim)
        self.dec_pred = nn.Linear(decoder_dim, patch_size * patch_size * 3)   # predict raw pixels/patch

    def random_masking(self, x):
        B, N, D = x.shape
        len_keep = int(N * (1 - self.mask_ratio))
        noise = torch.rand(B, N, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)          # random permutation per sample
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :len_keep]
        x_visible = torch.gather(x, 1, ids_keep.unsqueeze(-1).expand(-1, -1, D))

        mask = torch.ones(B, N, device=x.device)            # 1 = masked, 0 = visible/kept
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, 1, ids_restore)
        return x_visible, mask, ids_restore

    def forward(self, imgs):
        x = self.patch_embed(imgs) + self.pos_embed
        x_visible, mask, ids_restore = self.random_masking(x)
        for blk in self.encoder:
            x_visible = blk(x_visible)
        x_visible = self.enc_norm(x_visible)                # encoder ONLY ever sees visible patches

        # --- decoder: reinsert mask tokens at their original positions ---
        x_dec = self.decoder_embed(x_visible)
        B, len_keep, D = x_dec.shape
        N = self.num_patches
        mask_tokens = self.mask_token.expand(B, N - len_keep, -1)
        x_full = torch.cat([x_dec, mask_tokens], dim=1)
        x_full = torch.gather(x_full, 1, ids_restore.unsqueeze(-1).expand(-1, -1, D))  # unshuffle
        x_full = x_full + self.decoder_pos_embed
        for blk in self.decoder:
            x_full = blk(x_full)
        x_full = self.dec_norm(x_full)
        pred = self.dec_pred(x_full)                        # (B, N, patch_size^2 * 3)
        return pred, mask

    def loss(self, imgs, pred, mask):
        target = self._patchify(imgs)                       # (B, N, patch_size^2*3) ground-truth pixels
        loss = ((pred - target) ** 2).mean(dim=-1)           # per-patch MSE
        loss = (loss * mask).sum() / mask.sum()              # average over MASKED patches only
        return loss

    def _patchify(self, imgs):
        p = self.patch_size
        B, C, H, W = imgs.shape
        h = w = H // p
        x = imgs.reshape(B, C, h, p, w, p)
        x = x.permute(0, 2, 4, 3, 5, 1).reshape(B, h * w, p * p * C)
        return x


# --- demo ---
mae = MAE(img_size=224, patch_size=16, embed_dim=192, decoder_dim=96, mask_ratio=0.75)
imgs = torch.randn(2, 3, 224, 224)
pred, mask = mae(imgs)
loss = mae.loss(imgs, pred, mask)
print("MAE reconstruction loss (masked patches only):", loss.item(), "| masked fraction:", mask.mean().item())


MAE reconstruction loss (masked patches only): 1.3414455652236938 | masked fraction: 0.75



## 8. BEiT — ⭐⭐⭐ Masked *image modeling* (BERT-style, for images)

**Idea:** instead of reconstructing raw pixels (MAE), BEiT reconstructs **discrete visual
tokens**. A separately pretrained visual tokenizer (a discrete VAE) first converts each image
into a grid of *discrete token IDs* from a fixed visual vocabulary — like turning an image
into "visual words." Then, some patches of the input are masked, and a ViT encoder is trained
to predict the *token ID* of each masked patch (cross-entropy classification, exactly like
masked-language-modeling in BERT) using only the corrupted image as input.

**Why it matters:** shows the "BERT recipe" (mask + predict-the-token, not the pixel)
transfers to vision if you first discretize images into a learned vocabulary — a different
philosophy from MAE's regression-to-pixels approach.

**Used today for:** vision pretraining; historically influential on later masked-modeling
recipes (and conceptually related to how modern image/video generative tokenizers work).


In [11]:

class FakeVisualTokenizer(nn.Module):
    """
    Stand-in for a pretrained dVAE tokenizer: maps each patch to one of K discrete codebook ids.
    (In real BEiT this tokenizer is trained separately, ahead of time, and then frozen.)
    """
    def __init__(self, patch_dim=16 * 16 * 3, codebook_size=512, code_dim=32):
        super().__init__()
        self.encoder = nn.Linear(patch_dim, code_dim)
        self.codebook = nn.Parameter(torch.randn(codebook_size, code_dim))

    @torch.no_grad()
    def tokenize(self, patches):                            # patches: (B, N, patch_dim)
        z = self.encoder(patches)                            # (B, N, code_dim)
        dists = torch.cdist(z, self.codebook)                 # (B, N, K) distance to every codebook vector
        token_ids = dists.argmin(dim=-1)                      # nearest-codebook-entry id -> "visual word"
        return token_ids


class BEiT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, embed_dim=192, depth=4, num_heads=3,
                 codebook_size=512):
        super().__init__()
        self.patch_size = patch_size
        self.patch_embed = PatchEmbed(img_size, patch_size, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.mim_head = nn.Linear(embed_dim, codebook_size)   # predicts the masked patch's TOKEN ID

    def forward(self, imgs, tokenizer):
        p = self.patch_size
        B, C, H, W = imgs.shape
        patches = imgs.reshape(B, C, H // p, p, W // p, p).permute(0, 2, 4, 3, 5, 1).reshape(B, -1, p * p * C)
        target_ids = tokenizer.tokenize(patches)               # (B, N) ground-truth discrete tokens

        x = self.patch_embed(imgs) + self.pos_embed
        B, N, D = x.shape
        mask = (torch.rand(B, N, device=x.device) < 0.4)       # mask ~40% of patches
        mask_tok = self.mask_token.expand(B, N, D)
        x = torch.where(mask.unsqueeze(-1), mask_tok, x)       # replace masked patches with [MASK] embedding

        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        logits = self.mim_head(x)                               # (B, N, codebook_size)

        loss = F.cross_entropy(logits[mask], target_ids[mask])  # classification loss, masked positions only
        return loss


# --- demo ---
tokenizer = FakeVisualTokenizer(patch_dim=16 * 16 * 3, codebook_size=512, code_dim=32)
beit = BEiT(embed_dim=192, depth=4, num_heads=3, codebook_size=512)
imgs = torch.randn(2, 3, 224, 224)
loss = beit(imgs, tokenizer)
print("BEiT masked-token-classification loss:", loss.item())


BEiT masked-token-classification loss: 6.2510457038879395



## 9. SAM / SAM 2 — ⭐⭐⭐⭐⭐ Promptable segmentation

**Idea:** decompose segmentation into three reusable pieces:
1. **Image encoder** — a (heavy, ViT-based) backbone run *once* per image to get a dense
   image embedding.
2. **Prompt encoder** — turns a *prompt* (a point click, a box, or a rough mask) into an
   embedding of the same kind.
3. **Mask decoder** — a lightweight Transformer decoder that cross-attends the prompt
   embedding into the image embedding and outputs one or more candidate masks + confidence
   scores, fast enough to run interactively as the user adds prompts.

**SAM 2** extends this to **video**: it adds a *memory bank* + *memory attention* module so
that the mask decoder can also cross-attend into features/masks remembered from previous
frames, giving temporally-consistent promptable segmentation and tracking across a video,
still in (near) real time.

**Why it matters:** turned segmentation into an interactive, prompt-driven, zero-shot task
rather than a fixed-class, retrain-per-domain task.

**Used today for:** interactive image/video segmentation, annotation tooling, and as a
building block inside larger perception pipelines.


In [12]:

class PromptEncoder(nn.Module):
    """Encodes point/box prompts into the same embedding space as the image encoder."""
    def __init__(self, embed_dim=192):
        super().__init__()
        self.point_embed = nn.Linear(2, embed_dim)              # (x, y) coordinate -> embedding
        self.label_embed = nn.Embedding(2, embed_dim)            # 0 = negative click, 1 = positive click
        self.box_embed = nn.Linear(4, embed_dim)                 # (x0, y0, x1, y1) -> embedding

    def forward(self, points=None, point_labels=None, boxes=None):
        tokens = []
        if points is not None:
            tokens.append(self.point_embed(points) + self.label_embed(point_labels))
        if boxes is not None:
            tokens.append(self.box_embed(boxes))
        return torch.cat(tokens, dim=1)                          # (B, num_prompt_tokens, embed_dim)


class MaskDecoder(nn.Module):
    """Lightweight: a couple of cross-attention layers from prompt tokens into the image embedding."""
    def __init__(self, embed_dim=192, num_heads=3, num_masks=3, patch_grid=14):
        super().__init__()
        self.patch_grid = patch_grid
        self.mask_tokens = nn.Parameter(torch.zeros(1, num_masks, embed_dim))
        nn.init.trunc_normal_(self.mask_tokens, std=0.02)
        self.decoder_layers = nn.ModuleList([DETRDecoderLayer(embed_dim, num_heads) for _ in range(2)])
        self.iou_head = nn.Linear(embed_dim, 1)
        self.mask_mlp = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.ReLU(), nn.Linear(embed_dim, embed_dim))

    def forward(self, image_embed, prompt_tokens):
        B = image_embed.shape[0]
        queries = torch.cat([self.mask_tokens.expand(B, -1, -1), prompt_tokens], dim=1)
        for layer in self.decoder_layers:
            queries = layer(queries, image_embed)                # cross-attend INTO the (once-computed) image features
        mask_queries = queries[:, :self.mask_tokens.shape[1]]
        iou_scores = self.iou_head(mask_queries).squeeze(-1)                       # (B, num_masks) confidence
        mask_embeds = self.mask_mlp(mask_queries)                                  # (B, num_masks, D)
        # low-res mask logits via dot product with every image patch embedding
        low_res_masks = mask_embeds @ image_embed.transpose(1, 2)                  # (B, num_masks, num_patches)
        g = self.patch_grid
        low_res_masks = low_res_masks.view(B, -1, g, g)
        return low_res_masks, iou_scores


class SAM(nn.Module):
    def __init__(self, embed_dim=192):
        super().__init__()
        self.image_encoder = ViT(embed_dim=embed_dim, depth=4, num_heads=3, num_classes=0)  # run ONCE per image
        self.prompt_encoder = PromptEncoder(embed_dim)
        self.mask_decoder = MaskDecoder(embed_dim, num_heads=3, patch_grid=self.image_encoder.patch_embed.grid_size)

    def forward(self, image, points, point_labels):
        image_embed = self.image_encoder.forward_features(image)[:, 1:]   # patch tokens, drop CLS
        prompt_tokens = self.prompt_encoder(points=points, point_labels=point_labels)
        masks, iou_scores = self.mask_decoder(image_embed, prompt_tokens)  # cheap: re-run per NEW prompt
        return masks, iou_scores


# --- demo: click one positive point, get back 3 candidate masks ---
sam = SAM(embed_dim=192)
image = torch.randn(1, 3, 224, 224)
points = torch.tensor([[[100.0, 120.0]]])       # one (x, y) click
point_labels = torch.tensor([[1]])              # positive click
masks, iou_scores = sam(image, points, point_labels)
print("SAM candidate masks:", masks.shape, "| IoU confidence per mask:", iou_scores.shape)
print("SAM 2 adds a memory bank so THIS decoder can also attend to previous video frames' masks.")


SAM candidate masks: torch.Size([1, 3, 14, 14]) | IoU confidence per mask: torch.Size([1, 3])
SAM 2 adds a memory bank so THIS decoder can also attend to previous video frames' masks.



## 10. ViViT / Video ViT — ⭐⭐⭐ Spatiotemporal understanding

**Idea:** extend ViT from images to video. Two key ideas:
- **Tubelet embedding**: instead of embedding 2D patches, embed small 3D "tubelets"
  (`t × h × w` chunks of the video volume) directly with a 3D convolution — this fuses
  spatial *and* temporal information at the very first layer.
- **Factorized attention**: full joint spatiotemporal attention over every tubelet is
  expensive, so a common efficient variant *factorizes* it — first attend across space
  (within a frame), then attend across time (same spatial location, across frames) — instead
  of one full attention over the whole space-time volume.

**Why it matters:** the standard recipe for turning "ViT for images" into "Transformer for
video" — this same tubelet + factorized-attention idea reappears (in a diffusion context)
in the Video DiT section of the DiT notebook.

**Used today for:** video classification, action recognition, and as a conceptual precursor
to video generation backbones.


In [13]:

class TubeletEmbed(nn.Module):
    """3D conv: embeds (t x h x w) space-time tubelets directly -> fuses space+time at layer 1."""
    def __init__(self, tubelet_size=(2, 16, 16), in_chans=3, embed_dim=192):
        super().__init__()
        self.proj = nn.Conv3d(in_chans, embed_dim, kernel_size=tubelet_size, stride=tubelet_size)

    def forward(self, x):                       # x: (B, C, T, H, W)
        x = self.proj(x)                          # (B, D, T', H', W')
        B, D, T, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)           # (B, T'*H'*W', D)
        return x, (T, H, W)


class FactorizedSpatioTemporalBlock(nn.Module):
    """Attend across SPACE within each frame, then across TIME at each spatial location."""
    def __init__(self, dim, num_heads):
        super().__init__()
        self.spatial_norm = nn.LayerNorm(dim)
        self.spatial_attn = MHSA(dim, num_heads)
        self.temporal_norm = nn.LayerNorm(dim)
        self.temporal_attn = MHSA(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, 4.0)

    def forward(self, x, thw):
        T, H, W = thw
        B, N, D = x.shape

        # --- spatial attention: reshape so each frame's H*W tokens attend to each other ---
        x_sp = x.view(B * T, H * W, D)
        x_sp = x_sp + self.spatial_attn(self.spatial_norm(x_sp))
        x = x_sp.view(B, T, H * W, D)

        # --- temporal attention: reshape so each spatial location's T tokens attend across time ---
        x_t = x.permute(0, 2, 1, 3).reshape(B * H * W, T, D)
        x_t = x_t + self.temporal_attn(self.temporal_norm(x_t))
        x = x_t.view(B, H * W, T, D).permute(0, 2, 1, 3).reshape(B, N, D)

        x = x + self.mlp(self.norm2(x))
        return x


class VideoViT(nn.Module):
    def __init__(self, tubelet_size=(2, 16, 16), embed_dim=192, depth=4, num_heads=3, num_classes=10):
        super().__init__()
        self.tubelet_embed = TubeletEmbed(tubelet_size, embed_dim=embed_dim)
        self.blocks = nn.ModuleList([FactorizedSpatioTemporalBlock(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, video):                    # video: (B, C, T, H, W)
        x, thw = self.tubelet_embed(video)
        for blk in self.blocks:
            x = blk(x, thw)
        x = self.norm(x)
        return self.head(x.mean(dim=1))            # mean-pool over all space-time tokens


# --- demo ---
video_vit = VideoViT(tubelet_size=(2, 16, 16), embed_dim=192, depth=2, num_heads=3, num_classes=10)
video = torch.randn(1, 3, 8, 224, 224)   # 8 frames
logits = video_vit(video)
print("VideoViT logits:", logits.shape)


VideoViT logits: torch.Size([1, 10])



## Summary — how these models actually relate to each other

| Model | Built on | Core new idea | Where it shows up again |
|---|---|---|---|
| **ViT** | plain Transformer | patchify + self-attention on images | everywhere below |
| **DINOv2** | ViT | student/teacher self-distillation, no labels | feature extraction backbones |
| **CLIP** | ViT + text Transformer | softmax contrastive image-text loss | zero-shot classification, VLM vision towers |
| **SigLIP/2** | ViT + text Transformer | sigmoid (per-pair) contrastive loss | modern VLM vision towers |
| **Swin** | ViT | windowed + shifted attention, hierarchical | dense prediction backbones |
| **DETR** | ViT/CNN + Transformer decoder | learned queries + Hungarian matching | end-to-end detectors, DETR-style segmentation |
| **MAE** | ViT | mask ~75%, reconstruct pixels | cheap self-supervised pretraining |
| **BEiT** | ViT + discrete tokenizer | mask + predict *token id* (BERT-style) | masked-modeling pretraining |
| **SAM/SAM2** | ViT | image encoder + prompt encoder + mask decoder (+ memory for video) | interactive/promptable segmentation |
| **ViViT** | ViT | tubelet embedding + factorized space/time attention | video understanding, video-gen backbones |

**Reading order if revisiting:** ViT → DINOv2 → CLIP → SigLIP → Swin → DETR → MAE → BEiT →
SAM/SAM2 → ViViT. Each section only assumes you understand the ones above it.
